In [5]:
import requests
import json
import pandas as pd
"USE Bronze_Lakehouse"

def get_secret(name, vault_env="OUTBUILD_KEYVAULT_URL"):
    """Key Vault in Fabric, environment variable locally.

    Mirrors get_secret() in src/procore/procore_extract.py - credentials live in
    exactly one place and never in the notebook source.
    """
    import os
    vault = os.environ.get(vault_env)
    if vault:
        return notebookutils.credentials.getSecret(vault, name)
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Secret {name!r} not found. Set {vault_env} to your Key Vault URL, "
            f"or export {name} locally. See foundation/README.md."
        )
    return value

TOKEN = get_secret("OUTBUILD_API_TOKEN")

BASE_URL = "https://datahub.outbuild.com"
HEADERS = {
    "authorizationToken": TOKEN,
    "Content-Type": "application/json"
}

def fetch_all_pages(endpoint: str, data_key: str) -> list:
    """
    Handles both response shapes:
      - { "body": { "projects": [...], "hasNextPage": ... } }   (documented)
      - { "projects": [...], "hasNextPage": ... }                (actual)
    """
    results = []
    page = 1
    while True:
        response = requests.get(f"{BASE_URL}/{endpoint}?page={page}", headers=HEADERS)
        response.raise_for_status()
        raw = response.json()

        # Handle both response shapes
        body = raw.get("body", raw)  # fall back to top-level if no "body" key
        records = body.get(data_key, [])
        results.extend(records)

        print(f"  Page {page}: {len(records)} records")
        if not body.get("hasNextPage", False):
            break
        page += 1

    print(f"  Total: {len(results)}")
    return results

# ----------------------------------------------------------
# Fetch projects
# ----------------------------------------------------------
print("Fetching projects...")
projects_raw = fetch_all_pages("projects", "projects")

# Flatten: one row per project (no schedules column)
df_projects = pd.DataFrame([
    {k: v for k, v in p.items() if k != "schedules"}
    for p in projects_raw
])

# Flatten: one row per schedule, with project info attached
df_schedules = pd.DataFrame([
    {
        "project_id":   p["id"],
        "project_name": p["name"],
        **s
    }
    for p in projects_raw
    for s in p.get("schedules", [])
])

print("\n--- Projects ---")
display(df_projects[["id", "name", "type", "stage", "address", "procore_id", "created_at"]])

print("\n--- Schedules ---")
display(df_schedules[["project_id", "project_name", "id", "name", "is_current_schedule", "status", "productive"]])

# ----------------------------------------------------------
# Save to Bronze Lakehouse
# ----------------------------------------------------------
spark.createDataFrame(df_projects).write.format("delta").mode("overwrite").saveAsTable("bronze_outbuild_projects")
spark.createDataFrame(df_schedules).write.format("delta").mode("overwrite").saveAsTable("bronze_outbuild_schedules")

print("\n✅ Saved: bronze_outbuild_projects")
print("✅ Saved: bronze_outbuild_schedules")

StatementMeta(, 57c34af1-ae4d-439c-ae4e-3d007b2c2936, 7, Finished, Available, Finished, False)

Fetching projects...
  Page 1: 7 records
  Total: 7

--- Projects ---


SynapseWidget(Synapse.DataFrame, cca9391e-78f4-4a3d-b4f0-f31c0f070b3c)


--- Schedules ---


SynapseWidget(Synapse.DataFrame, a9ae97b2-21d5-4f5c-a8a5-54abf0358993)

UnsupportedOperationException: No default context found, please attach a lakehouse before running spark sql queries with partial namespaces.

In [6]:
import requests
import json
import pandas as pd

def get_secret(name, vault_env="OUTBUILD_KEYVAULT_URL"):
    """Key Vault in Fabric, environment variable locally.

    Mirrors get_secret() in src/procore/procore_extract.py - credentials live in
    exactly one place and never in the notebook source.
    """
    import os
    vault = os.environ.get(vault_env)
    if vault:
        return notebookutils.credentials.getSecret(vault, name)
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Secret {name!r} not found. Set {vault_env} to your Key Vault URL, "
            f"or export {name} locally. See foundation/README.md."
        )
    return value

TOKEN = get_secret("OUTBUILD_API_TOKEN")

BASE_URL = "https://datahub.outbuild.com"
HEADERS = {
    "authorizationToken": TOKEN,
    "Content-Type": "application/json"
}

# Test a few endpoints and print the raw response for each
endpoints_to_test = [
    ("projects",          "projects"),
    ("companies",         "companies"),
    ("user",              "user"),
]

for endpoint, data_key in endpoints_to_test:
    print(f"\n{'='*50}")
    print(f"GET /{endpoint}")
    print(f"{'='*50}")
    
    response = requests.get(f"{BASE_URL}/{endpoint}?page=1", headers=HEADERS)
    print(f"Status code: {response.status_code}")
    
    # Print the full raw JSON so we can see exactly what's returned
    try:
        raw = response.json()
        print(json.dumps(raw, indent=2))
    except Exception as e:
        print(f"Could not parse JSON: {e}")
        print(f"Raw text: {response.text}")

StatementMeta(, 57c34af1-ae4d-439c-ae4e-3d007b2c2936, 8, Finished, Available, Finished, False)


GET /projects
Status code: 200
{
  "projects": [
    {
      "id": 39780,
      "name": "Sauna Lounge",
      "planification_day": 1,
      "timezone": null,
      "country": "US",
      "currency": "USD",
      "budget": 0,
      "size_type": "ft2",
      "size": 0,
      "image": null,
      "type": "commercial",
      "stage": "started",
      "archive_reason": null,
      "pcr_goal": null,
      "pcc_goal": null,
      "task_creter": "duration",
      "activity_creter": "duration",
      "created_at": "2026-05-26T20:14:33.083Z",
      "updated_at": "2026-06-24T18:42:58.992Z",
      "address": "",
      "procore_id": null,
      "schedules": [
        {
          "id": 93970,
          "name": "Contract Schedule - R3",
          "description": null,
          "status": true,
          "order": 64,
          "productive": true,
          "created_at": "2026-05-26T20:14:33.130Z",
          "updated_at": "2026-06-25T02:07:51.936Z",
          "hours_per_day": 8,
          "hours_per_we

In [1]:
import requests
import json
import pandas as pd

def get_secret(name, vault_env="OUTBUILD_KEYVAULT_URL"):
    """Key Vault in Fabric, environment variable locally.

    Mirrors get_secret() in src/procore/procore_extract.py - credentials live in
    exactly one place and never in the notebook source.
    """
    import os
    vault = os.environ.get(vault_env)
    if vault:
        return notebookutils.credentials.getSecret(vault, name)
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Secret {name!r} not found. Set {vault_env} to your Key Vault URL, "
            f"or export {name} locally. See foundation/README.md."
        )
    return value

TOKEN = get_secret("OUTBUILD_API_TOKEN")

BASE_URL = "https://datahub.outbuild.com"
HEADERS = {
    "authorizationToken": TOKEN,
    "Content-Type": "application/json"
}

endpoints_to_test = [
    ("projects",   "projects"),
    ("companies",  "companies"),
    ("user",       "user"),
]

for endpoint, data_key in endpoints_to_test:
    print(f"\n{'='*50}")
    print(f"GET /{endpoint}  (status: ", end="")

    response = requests.get(f"{BASE_URL}/{endpoint}?page=1", headers=HEADERS)
    print(f"{response.status_code})")

    try:
        raw = response.json()
        body = raw.get("body", raw)  # handle both response shapes
        records = body.get(data_key)

        if records is None:
            # Single-object endpoints (e.g. /user) — wrap in a list
            records = [body] if body else []

        if records:
            df = pd.DataFrame(records)
            # If there are nested objects/lists, convert them to strings so the table renders cleanly
            for col in df.columns:
                if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                    df[col] = df[col].apply(lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x)
            display(df)
        else:
            print("  No records returned.")

    except Exception as e:
        print(f"  Could not parse response: {e}")
        print(f"  Raw text: {response.text}")

StatementMeta(, ef431bbc-9d49-4360-ba2c-3f079c9a3811, 3, Finished, Available, Finished, False)


GET /projects  (status: 200)


SynapseWidget(Synapse.DataFrame, 0ba3d993-4982-4af5-a046-a58f2da9d70f)


GET /companies  (status: 200)


SynapseWidget(Synapse.DataFrame, 469fe040-50bb-4983-9b2b-6d8205d18352)


GET /user  (status: 404)
  Could not parse response: Expecting value: line 1 column 1 (char 0)
  Raw text: Not Found


In [4]:
import requests
import json
import pandas as pd

# ----------------------------------------------------------
# Config
# ----------------------------------------------------------
def get_secret(name, vault_env="OUTBUILD_KEYVAULT_URL"):
    """Key Vault in Fabric, environment variable locally.

    Mirrors get_secret() in src/procore/procore_extract.py - credentials live in
    exactly one place and never in the notebook source.
    """
    import os
    vault = os.environ.get(vault_env)
    if vault:
        return notebookutils.credentials.getSecret(vault, name)
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(
            f"Secret {name!r} not found. Set {vault_env} to your Key Vault URL, "
            f"or export {name} locally. See foundation/README.md."
        )
    return value

TOKEN = get_secret("OUTBUILD_API_TOKEN")

BASE_URL = "https://datahub.outbuild.com"
HEADERS = {
    "authorizationToken": TOKEN,
    "Content-Type": "application/json"
}

# ----------------------------------------------------------
# Helper: paginated fetch
# ----------------------------------------------------------
def fetch_all_pages(endpoint: str, data_key: str) -> list:
    results = []
    page = 1
    while True:
        response = requests.get(f"{BASE_URL}/{endpoint}?page={page}", headers=HEADERS)
        response.raise_for_status()
        raw = response.json()
        body = raw.get("body", raw)
        records = body.get(data_key, [])
        results.extend(records)
        print(f"  Page {page}: {len(records)} records")
        if not body.get("hasNextPage", False):
            break
        page += 1
    print(f"  Total: {len(results)}")
    return results

# ----------------------------------------------------------
# Fetch & display all roadblocks
# ----------------------------------------------------------
print("Fetching all roadblocks...")
roadblocks = fetch_all_pages("roadblocks", "roadblocks")

df_roadblocks = pd.DataFrame(roadblocks)
display(df_roadblocks)

StatementMeta(, 8ea91012-52f0-422e-aa20-94b989b93503, 6, Finished, Available, Finished, False)

Fetching all roadblocks...
  Page 1: 160 records
  Total: 160


SynapseWidget(Synapse.DataFrame, b5249369-db67-41ac-a0fd-b5afd03fc7cb)